# Regression with a Flood Prediction Dataset - Sel Tahmini Veri Kümesiyle Regresyon Analizi

<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSZIXMBVBtHh8PQYhknbii9cJm59HxUR-7I-n8Cl2VNWw&s=10">

Bu çalışmada, çeşitli çevresel ve altyapısal faktörlerden yararlanılarak sel olasılığının tahmin edilmesi amaçlanmıştır. Veri seti üzerinde regresyon algoritmaları uygulanmış, elde edilen sonuçlar karşılaştırılarak en başarılı model belirlenmeye çalışılmıştır. Böylece sel riskinin önceden tahmin edilmesine yönelik veri odaklı bir yaklaşım geliştirilmiştir.

### Sütun Açıklamaları

**id**: Her satır için verilen benzersiz kayıt numarasıdır.

**MonsoonIntensity**: Muson yağmurlarının şiddetini ifade eder.

**TopographyDrainage**: Bölgenin arazi yapısı ve suyun tahliye olma durumunu gösterir.

**RiverManagement**: Nehirlerin yönetim ve kontrol durumunu ifade eder.

**Deforestation**: Ormansızlaşma seviyesini gösterir.

**Urbanization**: Bölgedeki şehirleşme düzeyini ifade eder.

**ClimateChange**: İklim değişikliğinin bölge üzerindeki etkisini gösterir.

**DamsQuality**: Barajların kalite ve güvenlik durumunu ifade eder.

**Siltation**: Nehir, göl veya barajlarda tortu (sediment) birikme seviyesini gösterir.

**AgriculturalPractices**: Tarımsal faaliyetlerin sel riski üzerindeki etkisini ifade eder.

**Encroachments**: Dere yatakları veya riskli alanlara yapılan yapılaşma seviyesini gösterir.

**IneffectiveDisasterPreparedness**: Afetlere karşı hazırlık seviyesinin yetersizliğini ifade eder.

**DrainageSystems**: Bölgedeki drenaj sistemlerinin yeterlilik durumunu gösterir.

**CoastalVulnerability**: Kıyı bölgelerinin sel ve taşkınlara karşı hassasiyetini ifade eder.

**Landslides**: Heyelan oluşma riskini veya heyelan etkisini gösterir.

**Watersheds**: Su toplama havzalarının sel riski üzerindeki etkisini ifade eder.

**DeterioratingInfrastructure**: Altyapının bozulma veya yetersizlik seviyesini gösterir.

**PopulationScore**: Bölgedeki nüfus yoğunluğu veya nüfus kaynaklı risk seviyesini ifade eder.

**WetlandLoss**: Sulak alan kaybı seviyesini gösterir.

**InadequatePlanning**: Yetersiz şehir ve çevre planlaması düzeyini ifade eder.

**PoliticalFactors**: Politik ve yönetsel faktörlerin sel riski üzerindeki etkisini gösterir.

**FloodProbability**: Sel meydana gelme olasılığını gösteren hedef değişkendir. Bu proje kapsamında tahmin edilmeye çalışılan değişkendir.

### Veri seti linki

https://www.kaggle.com/competitions/playground-series-s4e5/data

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s4e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s4e5/train.csv
/kaggle/input/competitions/playground-series-s4e5/test.csv


In [3]:
# Veri İşleme
import pandas as pd
import numpy as np

# Görselleştirme
import matplotlib.pyplot as plt
import seaborn as sns

# Uyarıları Gizleme
import warnings
warnings.filterwarnings("ignore")

# Veri Ön İşleme
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Regresyon Modelleri
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

# Model Değerlendirme
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Sonuç Tablosu İçin
from sklearn.metrics import make_scorer

In [4]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s4e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s4e5/train.csv
/kaggle/input/competitions/playground-series-s4e5/test.csv


In [5]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s4e5/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s4e5/test.csv')

print("Train boyutu:", train.shape)
print("Test boyutu :", test.shape)

train.head()

Train boyutu: (1117957, 22)
Test boyutu : (745305, 21)


,id,MonsoonIntensity,TopographyDrainage,RiverManagement,Deforestation,Urbanization,ClimateChange,DamsQuality,Siltation,AgriculturalPractices,...,DrainageSystems,CoastalVulnerability,Landslides,Watersheds,DeterioratingInfrastructure,PopulationScore,WetlandLoss,InadequatePlanning,PoliticalFactors,FloodProbability
0,0,5,8,5,8,6,4,4,3,3,...,5,3,3,5,4,7,5,7,3,0.445
1,1,6,7,4,4,8,8,3,5,4,...,7,2,0,3,5,3,3,4,3,0.450
2,2,6,5,6,7,3,7,1,5,4,...,7,3,7,5,6,8,2,3,3,0.530
3,3,3,4,6,5,4,8,4,7,6,...,2,4,7,4,4,6,5,7,5,0.535
4,4,5,3,2,6,4,4,3,3,3,...,2,2,6,6,4,1,2,3,5,0.415


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1117957 entries, 0 to 1117956
Data columns (total 22 columns):
 #   Column                           Non-Null Count    Dtype  
---  ------                           --------------    -----  
 0   id                               1117957 non-null  int64  
 1   MonsoonIntensity                 1117957 non-null  int64  
 2   TopographyDrainage               1117957 non-null  int64  
 3   RiverManagement                  1117957 non-null  int64  
 4   Deforestation                    1117957 non-null  int64  
 5   Urbanization                     1117957 non-null  int64  
 6   ClimateChange                    1117957 non-null  int64  
 7   DamsQuality                      1117957 non-null  int64  
 8   Siltation                        1117957 non-null  int64  
 9   AgriculturalPractices            1117957 non-null  int64  
 10  Encroachments                    1117957 non-null  int64  
 11  IneffectiveDisasterPreparedness  1117957 non-null 

In [7]:
train.isnull().sum()

id                                 0
MonsoonIntensity                   0
TopographyDrainage                 0
RiverManagement                    0
Deforestation                      0
Urbanization                       0
ClimateChange                      0
DamsQuality                        0
Siltation                          0
AgriculturalPractices              0
Encroachments                      0
IneffectiveDisasterPreparedness    0
DrainageSystems                    0
CoastalVulnerability               0
Landslides                         0
Watersheds                         0
DeterioratingInfrastructure        0
PopulationScore                    0
WetlandLoss                        0
InadequatePlanning                 0
PoliticalFactors                   0
FloodProbability                   0
dtype: int64

In [8]:
train.columns

Index(['id', 'MonsoonIntensity', 'TopographyDrainage', 'RiverManagement',
       'Deforestation', 'Urbanization', 'ClimateChange', 'DamsQuality',
       'Siltation', 'AgriculturalPractices', 'Encroachments',
       'IneffectiveDisasterPreparedness', 'DrainageSystems',
       'CoastalVulnerability', 'Landslides', 'Watersheds',
       'DeterioratingInfrastructure', 'PopulationScore', 'WetlandLoss',
       'InadequatePlanning', 'PoliticalFactors', 'FloodProbability'],
      dtype='object')

In [9]:
train['FloodProbability'].describe()

count    1.117957e+06
mean     5.044803e-01
std      5.102610e-02
min      2.850000e-01
25%      4.700000e-01
50%      5.050000e-01
75%      5.400000e-01
max      7.250000e-01
Name: FloodProbability, dtype: float64

In [10]:
x = train.drop(['id', 'FloodProbability'], axis=1)
y = train['FloodProbability']

In [11]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42
)

print(x_train.shape)
print(x_test.shape)

(894365, 20)
(223592, 20)


In [12]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

model1 = LinearRegression()

model1.fit(x_train, y_train)
tahmin1 = model1.predict(x_test)

print("R2 Score:", r2_score(y_test, tahmin1))
print("MAE:", mean_absolute_error(y_test, tahmin1))
print("RMSE:", np.sqrt(mean_squared_error(y_test, tahmin1)))

R2 Score: 0.8448773362840329
MAE: 0.015792471363760354
RMSE: 0.020080004658628893


In [13]:
from sklearn.linear_model import Ridge

model2 = Ridge()

model2.fit(x_train, y_train)
tahmin2 = model2.predict(x_test)

print("R2 Score:", r2_score(y_test, tahmin2))
print("MAE:", mean_absolute_error(y_test, tahmin2))
print("RMSE:", np.sqrt(mean_squared_error(y_test, tahmin2)))

R2 Score: 0.8448773363925368
MAE: 0.015792472328693095
RMSE: 0.020080004651606198


In [14]:
from sklearn.linear_model import Lasso

model3 = Lasso()

model3.fit(x_train, y_train)
tahmin3 = model3.predict(x_test)

print("R2 Score:", r2_score(y_test, tahmin3))
print("MAE:", mean_absolute_error(y_test, tahmin3))
print("RMSE:", np.sqrt(mean_squared_error(y_test, tahmin3)))

R2 Score: -2.041851132617012e-10
MAE: 0.04089184935658521
RMSE: 0.05098309336110274


In [15]:
from sklearn.linear_model import ElasticNet

model4 = ElasticNet()

model4.fit(x_train, y_train)
tahmin4 = model4.predict(x_test)

print("R2 Score:", r2_score(y_test, tahmin4))
print("MAE:", mean_absolute_error(y_test, tahmin4))
print("RMSE:", np.sqrt(mean_squared_error(y_test, tahmin4)))

R2 Score: -2.041851132617012e-10
MAE: 0.04089184935658521
RMSE: 0.05098309336110274


In [16]:
from sklearn.tree import DecisionTreeRegressor

model5 = DecisionTreeRegressor(random_state=42)

model5.fit(x_train, y_train)
tahmin5 = model5.predict(x_test)

print("R2 Score:", r2_score(y_test, tahmin5))
print("MAE:", mean_absolute_error(y_test, tahmin5))
print("RMSE:", np.sqrt(mean_squared_error(y_test, tahmin5)))

R2 Score: 0.05679790457739209
MAE: 0.03940496529392823
RMSE: 0.04951406253596388


## Sonuç

Uygulanan regresyon modelleri arasında en başarılı sonuçları Linear Regression ve Ridge Regression modelleri vermiştir. Her iki model de yaklaşık %84,49 R² skoru elde ederek sel olasılığındaki değişimin büyük bir kısmını açıklayabilmiştir. Ayrıca düşük MAE (≈0.0158) ve RMSE (≈0.0201) değerleri, tahminlerin gerçek değerlere oldukça yakın olduğunu göstermektedir. Buna karşılık Lasso, ElasticNet ve Decision Tree modelleri daha düşük performans göstermiştir. Elde edilen sonuçlar, veri setindeki değişkenler ile hedef değişken arasındaki ilişkinin büyük ölçüde doğrusal olduğunu ve bu nedenle Linear Regression ile Ridge Regression modellerinin bu problem için en uygun modeller olduğunu ortaya koymaktadır.